In [82]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [83]:
def load_and_explore_dataset(file_path):
        print("-" * 60)
        print("LOAD AND EXPLORE DATE")
        print("-" * 60)

        df = pd.read_csv(file_path)
        print('shape of the dataset')
        print(df.shape)
        print('\ncheck missing')
        print(df.isnull().sum())
        print('\nFirst five row')
        print(df.head())
        print('\nDescriptive statistics')
        print(df.describe())
        print('\ndataset info')
        print(df.info())
        print('\nCondition distribution')
        print(df['condition'].value_counts())
        print('\transmission distribution')
        print(df['transmission'].value_counts())

        return df

In [84]:
## Preprocessing
def preprocessing_data(df):
    print('\n' + "=" * 60)
    print("PREPROCESSING DATA")
    print("=" * 60)


    df_preprocessed = df.copy()
    df_preprocessed = df_preprocessed.dropna()


    label_encoder = {}
    df_columns = ['make', "model"]
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder = le
        print(f"\n{col} Encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label}: {i}")

    ordinal_encoder = OrdinalEncoder(
        categories=[
            ["Local Used", "Foreign Used", "Brand New"],
            ["Manual", "Automatic"]
        ]
    )

    df_preprocessed[["condition_encoded", "transmission_encoded"]] = (
        ordinal_encoder.fit_transform(
            df_preprocessed[["condition", "transmission"]]
        )
    )

    print("\nCondition Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[0]
    ):
        print(f"{label}: {i}")
        
    print("\nTransmission Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[1]
    ):
        print(f"{label}: {i}")
        
    print("Preprocessed data shape", df_preprocessed.shape)

    return df_preprocessed, label_encoder, ordinal_encoder

**Creating Data Label X and Y**

In [85]:
def featured_data(df_preprocessed):
    print('\n' + "=" * 60)
    print("FEATURED DATA")
    print("=" * 60)

    featured_columns = ["year", "make_encoded", "model_encoded", "condition_encoded", "transmission_encoded"]
    target_columns = ["price"]

    x = df_preprocessed[featured_columns]
    y = df_preprocessed[target_columns]

    print("\nFeatures shape:", df_preprocessed[featured_columns].shape)
    print("\nTarget shape:", df_preprocessed[target_columns].shape)
    print("\nFeature columns:", featured_columns)

    return x, y, featured_columns

**Spliting the Data for Training and Testing**

In [86]:
def split_data(x, y, test_size=0.2, random_state=42):
    print('\n' + "=" * 60)
    print("SPLITING DATA")
    print("=" * 60)

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", x_train.shape[0])
    print("\nTest set size:", x_test.shape[0])

    print("\nTraining set price range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    print("\nTest set price range {:.2f} - {:.2f}".format(
        float(y_test.values.min()), float(y_test.values.max())
    ))

    return x_train, x_test, y_train, y_test

**Scaling the data to standardize numerical values**

In [87]:
#making use of standard scaler
def scale_data(x_train, x_test):
    print('\n' + "=" * 60)
    print("SCALE DATA")
    print("=" * 60)

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.fit_transform(x_test)

    print("\nFeature scaled successfully")
    print("\nTraining set scaled shape:", x_train_scaled.shape)
    print("\nTesting set scaled shape:", x_test_scaled.shape)

    return x_train_scaled, x_test_scaled, scaler

**Training the model - Linear Regression**

In [97]:
def train_model(x_train_scale, y_train, featured_columns):
    print('\n' + "=" * 60)
    print("TRAIN MODEL")
    print("=" * 60)

    model = LinearRegression()
    model.fit(x_train_scale, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

    for feature, coef in zip(featured_columns, model.coef_.ravel()):
        print(f" {feature}: {coef:.2f}")
    print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")

    return model

In [98]:
def main():
    file_path = 'cleaned_jiji_car_dataset.csv'

    ## load and explore datset
    df = load_and_explore_dataset(file_path)


    ##preprocessed
    df_preprocessed, label_encoder, ordinal_encoder = preprocessing_data(df)

    ##features data
    x, y, featured_columns = featured_data(df_preprocessed)

    ##split train and test data
    x_train, x_test, y_train, y_test = split_data(x, y)

    ##scaled 
    x_train_scaled, x_test_scaled, scaler = scale_data(x_train, x_test)

    ##training model
    model = train_model(x_train_scaled, y_train, featured_columns)

In [99]:
if __name__ == "__main__":
    main()


------------------------------------------------------------
LOAD AND EXPLORE DATE
------------------------------------------------------------
shape of the dataset
(1773, 7)

check missing
title           0
make            0
model           0
year            0
condition       0
transmission    0
price           0
dtype: int64

First five row
                                  title    make         model  year  \
0                     Lexus ES 330 2006   Lexus        ES 330  2006   
1   Lexus RX 350 F Sport AWD 2018 Black   Lexus        RX 350  2018   
2            Lexus RX 350 FWD 2012 Gray   Lexus        RX 350  2012   
3  Toyota 4-Runner SR5 V8 4x4 2005 Blue  Toyota  4-Runner SR5  2005   
4              Toyota Sienna 2003 Green  Toyota        Sienna  2003   

      condition transmission       price  
0    Local Used    Automatic   6500000.0  
1    Local Used    Automatic  31000000.0  
2  Foreign Used    Automatic  20500000.0  
3    Local Used    Automatic   5800000.0  
4    Local Us